<a href="https://colab.research.google.com/github/Nanthakan670510759/229352-Labs/blob/main/Lab02_Data_Preprocessing_670510759%E0%B8%99%E0%B8%B1%E0%B8%99%E0%B8%97%E0%B8%81%E0%B8%B2%E0%B8%A3_%E0%B8%A3%E0%B8%AD%E0%B8%94%E0%B8%94%E0%B8%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #2

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

# For Fashion-MNIST
from tensorflow.keras.datasets import fashion_mnist

# For 20 Newsgroups
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Part 1: Marketing Campaign Dataset - Manual Data Preprocessing & Logistic Regression

### Load the Marketing Campaign Dataset ([Data Information](https://archive.ics.uci.edu/dataset/222/bank+marketing))

The data is related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be (`'yes'`) or not (`'no'`) subscribed.

In [2]:
bank_url = 'https://raw.githubusercontent.com/donlap/ds352-labs/main/bank.csv'

df = pd.read_csv(bank_url, sep=';', na_values=['unknown'])
df = df.drop(["emp.var.rate", "cons.price.idx", "cons.conf.idx",	"euribor3m", "nr.employed"], axis=1)
print("Shape of the dataset:", df.shape)
df.head()

Shape of the dataset: (41188, 16)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,no
1,57,services,married,high.school,NaN,no,no,telephone,may,mon,149,1,999,0,nonexistent,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,no


### Data Exploration

In [3]:
print("--- Missing Values Count ---")
print(df.isnull().sum()) #จำนวนแถวที่ตัวแปรหายไป

--- Missing Values Count ---
age               0
job             330
marital          80
education      1731
default        8597
housing         990
loan            990
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
y                 0
dtype: int64


In [4]:
print("--- Unique Values for Categorical Columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"\n'{col}' unique values:")
    print(df[col].value_counts(dropna=False)) # Include NaN counts

--- Unique Values for Categorical Columns ---

'job' unique values:
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
NaN                330
Name: count, dtype: int64

'marital' unique values:
marital
married     24928
single      11568
divorced     4612
NaN            80
Name: count, dtype: int64

'education' unique values:
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
NaN                     1731
illiterate                18
Name: count, dtype: int64

'default' unique values:
default
no     32588
NaN     8597
yes        3
Name: count, dtype: int64

'housing' unique values:
housing
yes    21576
no     18622
NaN      990
Name: count, dtype: int64


### Data Preprocessing

In [5]:
# Map target variable 'y' to 0 (no) and 1 (yes)
df['y_new'] = df['y'].map({'yes':1, 'no':0}) # Write your code here


# Drop 'duration' due to data leakage


# Define features (X) and target (y)
y=df['y_new']
X=df.drop(['y', 'y_new'], axis=1)
print(y)
print(X)

# Split the data BEFORE any transformations


# Print data shape



0        0
1        0
2        0
3        0
4        0
        ..
41183    1
41184    0
41185    0
41186    1
41187    0
Name: y_new, Length: 41188, dtype: int64
       age          job  marital            education default housing loan    contact month day_of_week  duration  campaign  pdays  previous     poutcome
0       56    housemaid  married             basic.4y      no      no   no  telephone   may         mon       261         1    999         0  nonexistent
1       57     services  married          high.school     NaN      no   no  telephone   may         mon       149         1    999         0  nonexistent
2       37     services  married          high.school      no     yes   no  telephone   may         mon       226         1    999         0  nonexistent
3       40       admin.  married             basic.6y      no      no   no  telephone   may         mon       151         1    999         0  nonexistent
4       56     services  married          high.school      no      n

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [7]:
X_train.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
40675,27,technician,single,university.degree,no,yes,no,cellular,sep,thu,530,1,3,1,success
23690,31,admin.,married,university.degree,no,yes,yes,cellular,aug,thu,12,5,999,0,nonexistent
13691,32,NaN,single,professional.course,no,yes,no,telephone,jul,thu,73,1,999,0,nonexistent
26428,33,blue-collar,divorced,basic.9y,no,yes,no,cellular,nov,thu,220,1,999,1,failure
30819,40,blue-collar,married,basic.9y,NaN,no,no,cellular,may,tue,335,1,999,0,nonexistent


In [8]:
y_train

,y_new
40675,1
23690,0
13691,0
26428,0
30819,0
...,...
5595,0
28282,0
36782,0
11675,0


We will apply `StandardScaler()`, `OrdinalEncoder()`, and `OneHotEncoder()` on a few selected columns.

**1. Numerical Feature: `age` and `campaign` (Standard Scaling)**

In [ ]:
num_cols_demo = ['age', 'campaign']

scaler = StandardScaler()
X_train[num_cols_demo]=scaler.fit_transform(X_train[num_cols_demo])
X_test[num_cols_demo]=scaler.transform(X_test[num_cols_demo])
X_test

In [ ]:
X_test.describe()

Let's take a look at the transformed `age` and `campaign` features and their statistics.

In [ ]:
print("\nOriginal X_train 'age' and 'campaign' head:")
print(X_train[num_cols_demo].head())
print("\nScaled X_train 'age' and 'campaign' head:")
print(pd.DataFrame(X_train_scaled_demo, columns=num_cols_demo, index=X_train.index).head())

print("\nMean of scaled 'age' (train):", X_train_scaled_demo[:, 0].mean())
print("Std Dev of scaled 'campaign' (train):", X_train_scaled_demo[:, 1].std())

**2. Ordinal Feature: `education` (Ordinal Encoding with Imputation)**

- **Imputation**

In [ ]:
ord_col_demo = ['education']

imputer_ord = SimpleImputer(strategy='most_frequent')
X_train[ord_col_demo]=imputer_ord.fit_transform(X_train[ord_col_demo])
X_test[ord_col_demo]=imputer_ord.transform(X_test[ord_col_demo])
X_train['education']

- **Ordinal Encoding**

In [ ]:
education_categories = [
    'illiterate', 'basic.4y', 'basic.6y', 'basic.9y', 'high.school',
    'professional.course', 'university.degree', 'masters', 'doctorate'
]

In [ ]:
ordinal_encoder = OrdinalEncoder(categories=[education_categories])
ord_col_demo = 'education'
X_train[ord_col_demo]=ordinal_encoder.fit_transform(X_train[[ord_col_demo]])
X_test[ord_col_demo]=ordinal_encoder.transform(X_test[[ord_col_demo]])
X_test


Let's take a look at the imputed and ordinal-encoded `education`.

In [ ]:
print("\nOriginal X_train 'education' head:")
print(X_train[ord_col_demo].iloc[20:25])
print("\nImputed X_train 'education' head (after imputer.transform):")
print(pd.DataFrame(X_train_imputed_ord_demo, columns=ord_col_demo, index=X_train.index).iloc[20:25])
print("\nOrdinal Encoded X_train 'education' head:")
print(pd.DataFrame(X_train_ord_encoded_demo, columns=ord_col_demo, index=X_train.index).iloc[20:25])

**3. Nominal Feature: `job` (One-Hot Encoding with Imputation)**

- **Imputation**

In [ ]:
nom_col_demo = ['job']

imputer_nom = SimpleImputer(strategy='most_frequent')
imputer_nom.fit(X_train[nom_col_demo])

X_train[nom_col_demo]= imputer_nom.fit_transform(X_train[nom_col_demo])
X_test[nom_col_demo]= imputer_nom.transform(X_test[nom_col_demo])
X_test

- **Nominal Encoding**

In [ ]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_onehot=onehot_encoder.fit_transform(X_train[['job']])
X_test_onehot=onehot_encoder.transform(X_test[['job']])
X_test_onehot.shape

In [ ]:
X_train=pd.concat([X_train.reset_index(drop=True), pd.DataFrame(X_train_onehot, columns=onehot_encoder.get_feature_names_out(['job']))],axis=1)
X_train

In [ ]:
X_test=pd.concat([X_test.reset_index(drop=True), pd.DataFrame(X_test_onehot, columns=onehot_encoder.get_feature_names_out(['job']))],axis=1)
X_test

In [ ]:
print("\nOriginal X_train 'job' head:")
print(X_train[nom_col_demo].iloc[40:45])
print("\nImputed X_train 'job' head (after imputer.transform):")
print(pd.DataFrame(X_train_imputed_nom_demo, columns=nom_col_demo, index=X_train.index).iloc[40:45])
print("\nOne-Hot Encoded X_train 'job' shape:", X_train_onehot_encoded_demo.shape)
print("First 5 rows of One-Hot Encoded X_train 'job':")
print(pd.DataFrame(X_train_onehot_encoded_demo, columns=onehot_encoder.get_feature_names_out(nom_col_demo), index=X_train.index).iloc[40:45])

### **Exercise 1: Apply All Preprocessing & Train Logistic Regression**

Now, it's your turn to apply these preprocessing steps to *all* relevant columns and then train a Logistic Regression model.

**Instructions:**

1.  Look at the Variable Table in [this link](https://archive.ics.uci.edu/dataset/222/bank+marketing).
2. Make lists for `numerical_features`, `ordinal_features`, and `nominal_features`.
3. Preprocess the features. It is safer to make a copy of `X_train` using:
   ```
   X_train_copy = X_train.copy()
   X_test_copy = X_test.copy()
   ```
   and preprocess `X_train_copy` instead.

   **For nominal features, concat the one-hot encoded features using [`pd.concat(..., axis=1)`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) and drop the old nominal features from the dataframe.**
4. Train Logistic Regression on the preprocessed `X_train_copy` and `y_train`.
5. Evaluate the Model:
    *   Make predictions on the preprocessed `X_test_copy`.
    *   Print `classification_report` ([Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)). What are the accuracy, average precision, average recall, and average f1-score?


In [ ]:
# --- YOUR CODE FOR EXERCISE 1 STARTS HERE ---

**Numerical Feature**

In [9]:
# num_cols_demo = ['age', 'job', 'marital','education','default','housing','loan','contact','month','day_of_week','duration','campaign','pdays','previous','poutcome']
num=['age','duration','campaign','pdays','previous']
cate=['job','marital','education','default','housing','loan','contact','month','day_of_week','poutcome']
onehot_encoder=OneHotEncoder(handle_unknown='ignore', sparse_output=False)
hot_train=onehot_encoder.fit_transform(X_train[cate])
hot_test=onehot_encoder.transform(X_test[cate])

cols=onehot_encoder.get_feature_names_out(cate)

df_hot_train=pd.DataFrame(hot_train, columns=cols, index=X_train.index)
df_hot_test=pd.DataFrame(hot_test, columns=cols, index=X_test.index)

scaler = StandardScaler()
X_train[num]=scaler.fit_transform(X_train[num])
X_test[num]=scaler.transform(X_test[num])
X_test

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
20410,1.146028,admin.,married,university.degree,no,yes,no,cellular,aug,tue,2.347440,-0.566706,0.192286,-0.345743,nonexistent
14503,0.283075,blue-collar,married,basic.9y,no,no,no,telephone,jul,tue,2.004043,0.154856,0.192286,-0.345743,nonexistent
12182,0.666610,retired,married,basic.4y,no,no,no,telephone,jul,tue,0.869677,-0.566706,0.192286,-0.345743,nonexistent
32139,-0.483995,services,single,high.school,no,yes,yes,cellular,may,fri,-0.635094,-0.566706,0.192286,-0.345743,nonexistent
12795,-0.771646,technician,married,professional.course,no,no,no,cellular,jul,tue,1.745531,-0.566706,0.192286,-0.345743,nonexistent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,0.858377,blue-collar,married,basic.9y,NaN,no,no,telephone,jun,thu,-0.727695,-0.566706,0.192286,-0.345743,nonexistent
2205,-1.059297,blue-collar,married,basic.9y,no,no,no,telephone,may,mon,-0.746987,0.876418,0.192286,-0.345743,nonexistent
23432,0.570726,technician,divorced,professional.course,no,yes,yes,cellular,aug,wed,-0.912898,0.876418,0.192286,-0.345743,nonexistent
23249,1.433680,management,married,university.degree,no,yes,no,cellular,aug,wed,-0.797146,-0.205925,0.192286,-0.345743,nonexistent


**Ordinal Feature**

In [10]:
ord_col_demo= ['age', 'job', 'marital','education','default','housing','loan','contact','month','day_of_week','duration','campaign','pdays','previous','poutcome']
imputer_ord = SimpleImputer(strategy='most_frequent')
X_train[ord_col_demo]=imputer_ord.fit_transform(X_train[ord_col_demo])
X_test[ord_col_demo]=imputer_ord.transform(X_test[ord_col_demo])
X_train

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
40675,-1.251064,technician,single,university.degree,no,yes,no,cellular,sep,thu,1.047163,-0.566706,-5.216856,1.687323,success
23690,-0.86753,admin.,married,university.degree,no,yes,yes,cellular,aug,thu,-0.951482,0.876418,0.192286,-0.345743,nonexistent
13691,-0.771646,admin.,single,professional.course,no,yes,no,telephone,jul,thu,-0.71612,-0.566706,0.192286,-0.345743,nonexistent
26428,-0.675762,blue-collar,divorced,basic.9y,no,yes,no,cellular,nov,thu,-0.148937,-0.566706,0.192286,1.687323,failure
30819,-0.004576,blue-collar,married,basic.9y,no,no,no,cellular,may,tue,0.294778,-0.566706,0.192286,-0.345743,nonexistent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5595,0.091308,admin.,single,basic.9y,no,no,yes,telephone,may,mon,0.425963,-0.566706,0.192286,-0.345743,nonexistent
28282,-1.251064,technician,single,basic.9y,no,yes,no,cellular,apr,wed,-0.658244,0.154856,0.192286,-0.345743,nonexistent
36782,-0.675762,management,married,university.degree,no,yes,no,cellular,jun,fri,-0.461466,0.876418,0.192286,-0.345743,nonexistent
11675,-1.059297,admin.,single,university.degree,no,yes,no,telephone,jun,fri,-0.959198,8.8136,0.192286,-0.345743,nonexistent


In [11]:
education_cate= ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y', 'high.school','professional.course', 'university.degree', 'masters', 'doctorate']
month_cate=['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov','dec']
day_cate=['mon','tue','wed','thu','fri','sat','sun']

ordinal_encoder = OrdinalEncoder(categories=[education_cate,month_cate,day_cate])
ord_col_demo = ['education','month','day_of_week']
X_train[ord_col_demo]=ordinal_encoder.fit_transform(X_train[ord_col_demo])
X_test[ord_col_demo]=ordinal_encoder.transform(X_test[ord_col_demo])
X_test

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
20410,1.146028,admin.,married,6.0,no,yes,no,cellular,7.0,1.0,2.34744,-0.566706,0.192286,-0.345743,nonexistent
14503,0.283075,blue-collar,married,3.0,no,no,no,telephone,6.0,1.0,2.004043,0.154856,0.192286,-0.345743,nonexistent
12182,0.66661,retired,married,1.0,no,no,no,telephone,6.0,1.0,0.869677,-0.566706,0.192286,-0.345743,nonexistent
32139,-0.483995,services,single,4.0,no,yes,yes,cellular,4.0,4.0,-0.635094,-0.566706,0.192286,-0.345743,nonexistent
12795,-0.771646,technician,married,5.0,no,no,no,cellular,6.0,1.0,1.745531,-0.566706,0.192286,-0.345743,nonexistent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,0.858377,blue-collar,married,3.0,no,no,no,telephone,5.0,3.0,-0.727695,-0.566706,0.192286,-0.345743,nonexistent
2205,-1.059297,blue-collar,married,3.0,no,no,no,telephone,4.0,0.0,-0.746987,0.876418,0.192286,-0.345743,nonexistent
23432,0.570726,technician,divorced,5.0,no,yes,yes,cellular,7.0,2.0,-0.912898,0.876418,0.192286,-0.345743,nonexistent
23249,1.43368,management,married,6.0,no,yes,no,cellular,7.0,2.0,-0.797146,-0.205925,0.192286,-0.345743,nonexistent


**Nominal Feature**

In [12]:
nom_col_demo = ['job','marital','default','housing','loan','contact','poutcome']

imputer_nom = SimpleImputer(strategy='most_frequent')
imputer_nom.fit(X_train[nom_col_demo])

X_train[nom_col_demo]= imputer_nom.fit_transform(X_train[nom_col_demo])
X_test[nom_col_demo]= imputer_nom.transform(X_test[nom_col_demo])
X_test

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
20410,1.146028,admin.,married,6.0,no,yes,no,cellular,7.0,1.0,2.34744,-0.566706,0.192286,-0.345743,nonexistent
14503,0.283075,blue-collar,married,3.0,no,no,no,telephone,6.0,1.0,2.004043,0.154856,0.192286,-0.345743,nonexistent
12182,0.66661,retired,married,1.0,no,no,no,telephone,6.0,1.0,0.869677,-0.566706,0.192286,-0.345743,nonexistent
32139,-0.483995,services,single,4.0,no,yes,yes,cellular,4.0,4.0,-0.635094,-0.566706,0.192286,-0.345743,nonexistent
12795,-0.771646,technician,married,5.0,no,no,no,cellular,6.0,1.0,1.745531,-0.566706,0.192286,-0.345743,nonexistent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,0.858377,blue-collar,married,3.0,no,no,no,telephone,5.0,3.0,-0.727695,-0.566706,0.192286,-0.345743,nonexistent
2205,-1.059297,blue-collar,married,3.0,no,no,no,telephone,4.0,0.0,-0.746987,0.876418,0.192286,-0.345743,nonexistent
23432,0.570726,technician,divorced,5.0,no,yes,yes,cellular,7.0,2.0,-0.912898,0.876418,0.192286,-0.345743,nonexistent
23249,1.43368,management,married,6.0,no,yes,no,cellular,7.0,2.0,-0.797146,-0.205925,0.192286,-0.345743,nonexistent


In [13]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_onehot=onehot_encoder.fit_transform(X_train[nom_col_demo])
X_test_onehot=onehot_encoder.transform(X_test[nom_col_demo])
X_test_onehot.shape

(12357, 25)

In [14]:
cols_to_ohe = ['job','marital','default','housing','loan','contact','poutcome']

X_train_ohe = pd.DataFrame(X_train_onehot,columns=onehot_encoder.get_feature_names_out(cols_to_ohe),index=X_train.index)
X_train = pd.concat([X_train.drop(columns=cols_to_ohe), X_train_ohe], axis=1)
X_train

,age,education,month,day_of_week,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,default_no,default_yes,housing_no,housing_yes,loan_no,loan_yes,contact_cellular,contact_telephone,poutcome_failure,poutcome_nonexistent,poutcome_success
40675,-1.251064,6.0,8.0,3.0,1.047163,-0.566706,-5.216856,1.687323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
23690,-0.86753,6.0,7.0,3.0,-0.951482,0.876418,0.192286,-0.345743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
13691,-0.771646,5.0,6.0,3.0,-0.71612,-0.566706,0.192286,-0.345743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
26428,-0.675762,3.0,10.0,3.0,-0.148937,-0.566706,0.192286,1.687323,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
30819,-0.004576,3.0,4.0,1.0,0.294778,-0.566706,0.192286,-0.345743,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5595,0.091308,3.0,4.0,0.0,0.425963,-0.566706,0.192286,-0.345743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
28282,-1.251064,3.0,3.0,2.0,-0.658244,0.154856,0.192286,-0.345743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
36782,-0.675762,6.0,5.0,4.0,-0.461466,0.876418,0.192286,-0.345743,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
11675,-1.059297,6.0,5.0,4.0,-0.959198,8.8136,0.192286,-0.345743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [16]:
X_test_ohe = pd.DataFrame(X_test_onehot,columns=onehot_encoder.get_feature_names_out(cols_to_ohe),index=X_test.index)
X_test = pd.concat([X_test.drop(columns=cols_to_ohe), X_test_ohe], axis=1)
X_test

,age,education,month,day_of_week,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,default_no,default_yes,housing_no,housing_yes,loan_no,loan_yes,contact_cellular,contact_telephone,poutcome_failure,poutcome_nonexistent,poutcome_success
20410,1.146028,6.0,7.0,1.0,2.34744,-0.566706,0.192286,-0.345743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
14503,0.283075,3.0,6.0,1.0,2.004043,0.154856,0.192286,-0.345743,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
12182,0.66661,1.0,6.0,1.0,0.869677,-0.566706,0.192286,-0.345743,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
32139,-0.483995,4.0,4.0,4.0,-0.635094,-0.566706,0.192286,-0.345743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
12795,-0.771646,5.0,6.0,1.0,1.745531,-0.566706,0.192286,-0.345743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,0.858377,3.0,5.0,3.0,-0.727695,-0.566706,0.192286,-0.345743,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2205,-1.059297,3.0,4.0,0.0,-0.746987,0.876418,0.192286,-0.345743,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
23432,0.570726,5.0,7.0,2.0,-0.912898,0.876418,0.192286,-0.345743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
23249,1.43368,6.0,7.0,2.0,-0.797146,-0.205925,0.192286,-0.345743,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [17]:
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

In [18]:
log=LogisticRegression(max_iter=1000)
log.fit(X_train_copy, y_train)
log

LogisticRegression(max_iter=1000)

In [19]:
y_pred=log.predict(X_test_copy)
y_pred

array([1, 0, 0, ..., 0, 0, 0])

In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.98      0.95     10951
           1       0.69      0.35      0.47      1406

    accuracy                           0.91     12357
   macro avg       0.81      0.67      0.71     12357
weighted avg       0.90      0.91      0.89     12357



## Part 2: Fashion-MNIST Dataset - Image Classification

### Load Fashion-MNIST Dataset

The Fashion-MNIST dataset consists of 28x28 grayscale images of fashion items.

In [ ]:
(fm_X_train, fm_y_train), (fm_X_test, fm_y_test) = fashion_mnist.load_data()

print(f"Fashion-MNIST Train data shape: {fm_X_train.shape}")
print(f"Fashion-MNIST Train labels shape: {fm_y_train.shape}")
print(f"Fashion-MNIST Test data shape: {fm_X_test.shape}")
print(f"Fashion-MNIST Test labels shape: {fm_y_test.shape}")

In [ ]:
print(f"First image {fm_X_train[0]}")
print(f"First label {fm_y_train[0]}")

### Visualize Fashion-MNIST Images

Let's see what these images look like.

In [ ]:
fashion_mnist_class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Visualize the images
## Write your code here



### **Exercise 2: Preprocessing Images (Flatten and Scale)**

Images are 2D arrays (matrices of pixels) and pixel values are integers from 0-255. For Logistic Regression, we need:
*  **Flattening:** Convert each 28x28 image into a 1D array of 784 features.
*  **Scaling:** Normalize pixel values from [0, 255] to [0, 1].

**Instructions:**

1.   **Flatten:** Use the `.reshape()` method (see [documentation](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.reshape.html)). For `fm_X_train_binary` (shape `(num_samples, 28, 28)`), you want to reshape it to `(num_samples, 28*28)`.
2.  **Scale:** Divide the flattened pixel values by 255.0 to get values between 0 and 1.
3.   **Train Logistic Regression:**
    *   Initialize `LogisticRegression(solver='saga')`. `saga` is a good solver when both number of samples and number of features are large.
    *   Fit the model on your *processed* `fm_X_train_scaled` and `fm_y_train`.
4.   **Make Predictions:** Use `predict()` to make predictions on the *processed* `fm_X_test_scaled`.
5.   **Print Classification Report:** Print `classification_report` ([Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)). What are the accuracy, average precision, average recall, and average f1-score?
6.   **Visualize Misclassifications:**
    *   Find the indices in `fm_X_test_binary` where your model made incorrect predictions (i.e., `fm_y_pred != fm_y_test`).
    *   Select 5 of these misclassified images.
    *   Plot these images (using `plt.imshow`). For each image, print its true label and its predicted label.

In [ ]:
# --- YOUR CODE FOR EXERCISE 2 STARTS HERE ---





## Part 3: 20 Newsgroups Dataset - Text Classification

### Load 20 Newsgroups Dataset

The 20 newsgroups dataset comprises around 18000 newsgroups posts on 20 topics.

In [ ]:
news_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
news_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

X_train_news, y_train_news = news_train.data, news_train.target
X_test_news, y_test_news = news_test.data, news_test.target

print(f"Number of training documents: {len(X_train_news)}")
print(f"Number of test documents: {len(X_test_news)}")
print(f"Categories: {news_train.target_names}")

### Explore Sample Document

In [ ]:
# Print the first document and its class
## Write your code here



### Preprocessing: Text Vectorization Demonstration with `TfidfVectorizer`

$$
\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)
$$

Where:

$$
\text{TF}(t, d) = \frac{\text{number of word }t\text{ in } d}{\text{number of words in } d} \quad \text{ and } \quad
\text{IDF}(t, D) = \log\left(\frac{\text{total number of documents}}{\text{number of documents that contain word }t}\right).
$$

In [ ]:
sample_sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]

vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform the sample sentences
sample_vec_output_sparse = # Write your code here

sample_vec_output_dense = sample_vec_output_sparse.toarray()

print(vectorizer.vocabulary_)
print(vectorizer.get_feature_names_out())
print(sample_vec_output_dense)

### **Exercise 3: Apply TF-IDF Vectorization to Full Dataset**

Now, apply `TfidfVectorizer` to the actual training and testing datasets for the 20 Newsgroups classification task.

**Instructions:**

1.  **Initialize `TfidfVectorizer`:**
    *   Initialize `TfidfVectorizer`. Use `stop_words='english'` to remove common words.
2.  **Fit and Transform Training Data:**
    *   Call `fit_transform()` on `X_train_news` to learn the vocabulary and transform the training text into TF-IDF features. Store the result in `X_train_vec`.
3.  **Transform Test Data:**
    *   Call `transform()` on `X_test_news` using the *already fitted* vectorizer. Store the result in `X_test_vec`. **Crucially, do not call `fit_transform()` on the test data!** This would cause data leakage.
4.  **Initialize Logistic Regression:**
    *   Initialize `LogisticRegression(solver='saga')`. `saga` is a good solver when both number of samples and number of features are large.
5.  **Train the Model:**
    *   Fit the model on your `X_train_vec` and `y_train_news`.
6.  **Make Predictions:**
    *   Make predictions using `predict()` on the `X_test_vec`.
7.  **Evaluate the Model:**
    *   Print `classification_report` ([Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)). What are the accuracy, average precision, average recall, and average f1-score?

In [ ]:
# --- YOUR CODE FOR EXERCISE 3 STARTS HERE ---


